# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Version: {metadata['version']}")
print(f"Keywords: {metadata['keywords']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id

record_sets = dataset.record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# If there is at least one record set, show fields for it
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nFields for Record Set @id: {first_record_set_id}")
    fields = dataset.fields(record_set=first_record_set_id)
    for f in fields:
        print(f"  Field @id: {f['@id']}, name: {f.get('name', '<no name>')}, dataType: {f.get('dataType', '<no dataType>')}")
    # Optionally, show a few records
    print("\nExample records:")
    for x in dataset.records(record_set=first_record_set_id):
        print(x)
        break  # Print just one to see structure

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets_info = dataset.record_sets()
record_sets_ids = [rs['@id'] for rs in record_sets_info]

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in RecordSet {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print("No records available for this record set.")

# For demonstration, select the first available record set and show its columns
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing Record Set: {example_record_set_id}")
    print("Columns:", dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use the GAD-7 (Generalized Anxiety Disorder) score as a numeric field, referenced by its column `@id`, and group by level of education.

In [ ]:
# Find GAD-7 score column @id and level_of_education column @id
import re

df = dataframes[example_record_set_id]

# Identify columns that likely correspond to GAD-7 and education by regex
gad7_candidates = [col for col in df.columns if re.search('gad[-_]?7|GAD[-_]?7', col, re.IGNORECASE)]
edu_candidates = [col for col in df.columns if re.search('education|level_of_education', col, re.IGNORECASE)]

print("GAD-7 score fields:", gad7_candidates)
print("Education fields:", edu_candidates)

# For demonstration, pick first match or define manually
numeric_field = gad7_candidates[0] if gad7_candidates else df.columns[0]
group_field = edu_candidates[0] if edu_candidates else None

# Filter records: GAD-7 score above threshold
threshold = 10
# Remove non-numeric, safely
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
mean = filtered_df[numeric_field].mean()
std = filtered_df[numeric_field].std()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouped analysis by education field
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data: mean {numeric_field} by {group_field}")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Boxplot for GAD-7 by education
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading the Kilifi Mental Health Survey FAIR² dataset via `mlcroissant` and exploring the metadata, record sets, and fields.
- We extracted all available record sets by their `@id`, explored GAD-7 scores and performed filtering and normalization.
- Grouping by education level highlighted possible differences in average anxiety scores.
- Visualizations revealed distribution characteristics for the selected mental health measure.

Further analysis can leverage other psychological scores and demographic fields as referenced by their `@id`.